# **금융경제학**

- 금융경제학 (박기영 저, 시그마프레스) 교재에 사용된 데이터/모형/그래프 관련 작업을 수행하는 python notebook 파일임: https://github.com/FinancialEconomicsPython/book
- python 코드는 구글 코랩에서 사용하는 것을 기준으로 작성되었음.
- 데이터 파일이 필요한 경우 위치: https://drive.google.com/drive/folders/1sArqUZKnxWtkNtHe31iD1w-2xCVEhTj0?usp=share_link
- date: 2025/3/22, updated: 2026/2/7

# 사전준비

## 실행 방법
- **아래 셀 하나만 실행**하면 준비가 끝납니다 — 리포 경로 자동 감지, 라이브러리 설치, 그래프 스타일, 한글 폰트, NBER 경기침체 데이터 로딩까지 한 번에 처리합니다.
- 준비할 것은 **한국은행 ECOS API 키** 하나뿐입니다. 두 가지 방법 중 하나:
  1. **(권장)** Colab 왼쪽 🔑(Secrets)에 이름 `ECOS_API_KEY`로 키를 저장하세요. 노트북에 키가 남지 않아 안전합니다.
  2. 아래 셀의 `ecos_key="..."`에 직접 입력하세요.
  - ECOS open API 키 신청: https://ecos.bok.or.kr/api/#/
- 리포를 git clone 했거나(README 방법 A) Colab에서 실행하면 경로(BASE/UTILS/FIGS/DATA)는 **자동으로 잡히므로 직접 지정할 필요가 없습니다.**
- 준비 로직의 상세 구현은 `utils/bootstrap.py`에 있습니다.


In [ ]:
# ============================================================
# 부트스트랩 — 모든 장 공통 (상세 구현: utils/bootstrap.py)
# ============================================================
import os, sys

# 리포 루트 자동 감지: (1) 이미 clone됨 → (2) Google Drive(저자) → (3) 자동 clone → (4) 로컬
if os.path.isdir("/content") and not os.path.exists("/content/resources"):
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except Exception:
        pass

_DRIVE = "/content/drive/MyDrive/Colab Notebooks/book_FinancialEconomics"
if os.path.exists("/content/resources"):
    ROOT = "/content/resources"          # 독자: git clone 방식(README 방법 A)
elif os.path.isdir(_DRIVE):
    ROOT = _DRIVE                        # 저자: Google Drive 방식
elif os.path.isdir("/content"):
    os.system("git clone -q https://github.com/FinancialEconomicsPython/resources.git /content/resources")
    ROOT = "/content/resources"          # 독자: 자동 clone
else:
    ROOT = os.getcwd()                   # 로컬 실행

sys.path.insert(0, os.path.join(ROOT, "utils"))
from bootstrap import init

# ECOS API 키: Colab Secrets(🔑)에 'ECOS_API_KEY' 저장을 권장. 없으면 아래에 직접 입력.
env = init(globals(), root=ROOT, ecos_key="YOUR_ECOS_API_KEY_HERE")


# Main

In [ ]:
def cumret(df, NetReturn=True, InPercentage=True):
    '''
    function for converting return to cumulative one
    df: net return data (not gross return)
    InPercentage=True: denoted as 3.14%, not 0.0314
    '''
    if InPercentage==True:
        if NetReturn==True:
            lnret = np.log(1+df/100)
        else:
            lnret = np.log(df/100)
    else:
        if NetReturn==True:
            lnret = np.log(1+df)
        else:
            lnret = np.log(df)

    cum_ret = np.exp(lnret.cumsum())-1

    return cum_ret

In [ ]:
def plot_ret(df, title, InPercentage=True):
    '''
    function for plotting the time series of returns
    df: dataframe of the returns
    title: figure title
    InPercentage=True: denoted as 3.14%, not 0.0314
    '''
    fig, ax = plt.subplots(1,1,figsize=(9,6))

    if InPercentage==True:
        df = df*100
        ax.set_ylabel('(%)')

    ax.plot(df)
    ax.set_title(title)
    ax.grid()
    ax.legend(df.columns)

    save_fig(title)

## Importing data from Fama-French website

In [ ]:
import os
import pandas_datareader as pdr
from datetime import date
import pandas_datareader.data as web  # module for reading datasets directly from the web
from pandas_datareader.famafrench import get_available_datasets

In [ ]:
start_date = '1926-07-01'
end_date = '2023-12-31'

FF3 = web.DataReader('F-F_Research_Data_Factors', 'famafrench',start=start_date,end=end_date)[0]
FF3.index.names=['mdate']
FF3['Mkt'] = FF3['Mkt-RF'] + FF3['RF']
FF3.head(2)

,Mkt-RF,SMB,HML,RF,Mkt
mdate,,,,,
1926-07,2.890,-2.550,-2.390,0.220,3.110
1926-08,2.640,-1.140,3.810,0.250,2.890


In [ ]:
def generate_EP_statistics(df, start, end):
  '''
  calcaulate the average of market portfolio, riskfree rate, and equity premium + Sharpe ratio

  input:
    df: return data
    start: starting date
    end: ending date
  '''
  avg_rm = df.loc[start:end,'Mkt'].mean()*12
  avg_rf = df.loc[start:end,'RF'].mean()*12
  AnnualRe = df.loc[start:end,'Mkt-RF'].mean()*12
  SigmaRe = df.loc[start:end,'Mkt-RF'].std()*np.sqrt(12)
  SR = AnnualRe/SigmaRe

  var_name = start+':'+end
  index = ['avg. market return','avg. riskfree rate','avg. equity premium','std','Sharpe ratio']
  df_table = pd.DataFrame({var_name:[avg_rm,avg_rf,AnnualRe,SigmaRe,SR]},index=index)
  return df_table

In [ ]:
df1 = generate_EP_statistics(FF3,'1926','2023')
print(df1)
df2 = generate_EP_statistics(FF3,'1946','2023')
print(df2)
df3 = generate_EP_statistics(FF3,'2000','2023')
print(df3)
df4 = generate_EP_statistics(FF3,'2008','2023')
print(df4)

                     1926:2023
avg. market return      11.365
avg. riskfree rate       3.211
avg. equity premium      8.154
std                     18.504
Sharpe ratio             0.441
                     1946:2023
avg. market return      11.598
avg. riskfree rate       3.763
avg. equity premium      7.834
std                     15.021
Sharpe ratio             0.522
                     2000:2023
avg. market return       8.310
avg. riskfree rate       1.625
avg. equity premium      6.685
std                     16.026
Sharpe ratio             0.417
                     2008:2023
avg. market return      10.907
avg. riskfree rate       0.843
avg. equity premium     10.064
std                     16.731
Sharpe ratio             0.602


## Mehra and Prescott (1985)

In [ ]:
# 모수 값
u = 1.06    # 호황
d = 0.98    # 불황
beta = 0.99 # 주관적 시선호율

gamma = 10  # 상대적 위험기피계수
x = -0.3    # 지속성(persistence) 모수

puu = (1 + x) / 2
pud = (1 - x) / 2
pdu = (1 - x) / 2
pdd = (1 + x) / 2

buu = beta * puu * u ** (-gamma)
bud = beta * pud * d ** (-gamma)
bdu = beta * pdu * u ** (-gamma)
bdd = beta * pdd * d ** (-gamma)

In [ ]:
# perpetuity
A = np.array([[1 - buu, -bud],
              [-bdu, 1 - bdd]])

pp = np.linalg.inv(A).dot(np.array([buu + bud,
                                     bdu + bdd]))

# 1-기간 무위험 이자율
Rf = 1 / np.array([buu + bud, bdu + bdd])

# p/d or p/c 비율
cuu = beta * puu * u ** (1 - gamma)
cud = beta * pud * d ** (1 - gamma)
cdu = beta * pdu * u ** (1 - gamma)
cdd = beta * pdd * d ** (1 - gamma)

A = np.array([[1 - cuu, -cud],
              [-cdu, 1 - cdd]])
pc = np.linalg.inv(A).dot(np.array([cuu + cud,
                                     cdu + cdd]))

# expected return
ER = np.array([puu * (pc[0] + 1) * u / pc[0] + pud * (pc[1] + 1) * d / pc[0],
               pdu * (pc[0] + 1) * u / pc[1] + pdd * (pc[1] + 1) * d / pc[1]])
ERp = np.array([puu * (pp[0] + 1) / pp[0] + pud * (pp[1] + 1) / pp[0],
                pdu * (pp[0] + 1) / pp[1] + pdd * (pp[1] + 1) / pp[1]])

Edc = np.array([puu * u + pud * d,
                pdu * u + pdd * d])
phi = (Edc[1] - Edc[0]) / (d - u)

# actual returns
pc1 = np.array([1, 1])[:, np.newaxis] * pc
dc1 = np.array([u, d])[:, np.newaxis]
pc0 = pc[:, np.newaxis] * np.array([1, 1])
Rs = ((pc1 + 1) * dc1) / pc0
Rs = (Rs - 1) * 100

pp1 = np.array([1, 1])[:, np.newaxis] * pp
pp0 = pp[:, np.newaxis] * np.array([1, 1])
Rp = (pp1 + 1) / pp0
Rp = (Rp - 1) * 100

In [ ]:
# ============================================================
# 출력
# ============================================================
shortoutput = True

if shortoutput:
    print(f'상대적 위험기피 계수 (gamma): {gamma:.2g}')
    print(f'persistence (x):              {x:.2g}')
    print()

    header = f"{'':12s} {'bond price':>10s} {'p/c':>8s} {'Rf (%)':>8s} {'ER-Rf (%)':>10s} {'ERp-Rf (%)':>11s}"
    print(header)
    for i, state in enumerate(['호황(u)', '불황(d)']):
        row = np.array([pp[i], pc[i], Rf_pct[i],
                        100*(ER[i] - Rf[i]), 100*(ERp[i] - Rf[i])])
        print(f'{state:12s} {row[0]:>10.4f} {row[1]:>8.4f} {row[2]:>8.4f} {row[3]:>10.4f} {row[4]:>11.4f}')

    print()
    print('stock / bond excess return (2x2) — 행=현재상태, 열=다음상태')
    print('           u→u       u→d  |   u→u       u→d')
    print('           d→u       d→d  |   d→u       d→d')
    for i, state in enumerate(['호황(u)', '불황(d)']):
        xs_s = Rs_pct[i] - Rf_pct[i]   # 주식 초과수익
        xs_p = Rp_pct[i] - Rf_pct[i]   # 채권 초과수익
        print(f'{state}: {np.round(xs_s, 4)}  |  {np.round(xs_p, 4)}')

else:
    print(f'[u, d, beta, gamma, x, phi] = {[u, d, beta, gamma, x, round(phi,6)]}')
    print()
    header = f"{'':8s} {'g(%)':>6s} {'E[g](%)':>8s} {'pp':>8s} {'pc':>8s} {'Rf(%)':>7s} {'ER(%)':>7s} {'ERp(%)':>8s} {'ER-Rf':>7s} {'ERp-Rf':>8s}"
    print(header)
    for i, (state, gi) in enumerate(zip(['호황(u)', '불황(d)'], [u, d])):
        row = np.array([(gi-1)*100, (Edc[i]-1)*100, pp[i], pc[i],
                        Rf_pct[i], 100*(ER[i]-1), 100*(ERp[i]-1),
                        100*(ER[i]-Rf[i]), 100*(ERp[i]-Rf[i])])
        print(f'{state:8s} {row[0]:>6.2f} {row[1]:>8.4f} {row[2]:>8.4f} {row[3]:>8.4f} '
              f'{row[4]:>7.4f} {row[5]:>7.4f} {row[6]:>8.4f} {row[7]:>7.4f} {row[8]:>8.4f}')

    print()
    print('stock / bond actual & excess return — 행=현재상태, 열=다음상태')
    for i, state in enumerate(['호황(u)', '불황(d)']):
        xs_s = Rs_pct[i] - Rf_pct[i]
        xs_p = Rp_pct[i] - Rf_pct[i]
        print(f'{state}: Rs={np.round(Rs_pct[i],4)}  xs_s={np.round(xs_s,4)}  '
              f'Rp={np.round(Rp_pct[i],4)}  xs_p={np.round(xs_p,4)}')

상대적 위험기피 계수 (gamma): 10
persistence (x):              -0.3

             bond price      p/c   Rf (%)  ER-Rf (%)  ERp-Rf (%)
호황(u)            6.5124   6.9509   1.9315     3.6765      2.5547
불황(d)            5.4234   5.8931  27.6484     5.4305      3.8417

stock / bond excess return (2x2) — 행=현재상태, 열=다음상태
           u→u       u→d  |   u→u       u→d
           d→u       d→d  |   d→u       d→d
호황(u): [19.318  3.188]  |  [13.424 -3.298]
불황(d): [  4.571 -13.019]  |  [10.869 -9.21 ]
